# Phase 0 — Thyroid Ultrasound Dataset Audit

This notebook audits the mounted datasets before any model training. It checks:

- the actual Kaggle paths;
- image and mask pairing;
- split overlap and duplicate hashes;
- TN5000 XML labels and bounding boxes;
- TNUI-2021 labels and segmentation files;
- ThyUS2Path CSV metadata;
- image resolutions and mask areas;
- representative visual examples.

**Do not train a model until this audit is complete.** The output manifest files will later define the official experiment splits and prevent data leakage.

## Expected mounted structure

The notebook is configured for the paths supplied by the author:

```text
/kaggle/input/datasets/jonbon8397/
/kaggle/input/datasets/dasmehdixtr/
```

If Kaggle uses a different mount path, edit only the `INPUT_ROOT` variable in the next cell.

In [ ]:
from pathlib import Path
import os
import re
import json
import hashlib
import random
import warnings
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

INPUT_ROOT = Path("/kaggle/input/datasets")
AUTHOR_ROOT = INPUT_ROOT / "jonbon8397"
DDTI_MIRROR_ROOT = INPUT_ROOT / "dasmehdixtr" / "ddti-thyroid-ultrasound-images"
OUTPUT_ROOT = Path("/kaggle/working/thyroid_audit")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PATHS = {
    "thyus2path": AUTHOR_ROOT / "thyus2path",
    "tn5000": AUTHOR_ROOT / "tn5000" / "TN5000_forReview",
    "thyroid_dataset": AUTHOR_ROOT / "tn3k-data" / "Thyroid Dataset",
    "tn3k": AUTHOR_ROOT / "tn3k-data" / "Thyroid Dataset" / "tn3k",
    "ddti": AUTHOR_ROOT / "tn3k-data" / "Thyroid Dataset" / "DDTI dataset" / "DDTI",
    "tnui": AUTHOR_ROOT / "tnui-2021" / "TNUI-2021-" / "thyroid_data",
    "ddti_mirror": DDTI_MIRROR_ROOT,
}

print("Input root:", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)
for name, path in PATHS.items():
    print(f"{name:18s} | {'OK' if path.exists() else 'MISSING'} | {path}")

## 1. File inventory

This cell counts files by extension and prints the first-level structure. It is intentionally lightweight and does not load every image.

In [ ]:
def file_inventory(path, max_rows=30):
    if not path.exists():
        return pd.DataFrame(columns=["extension", "count"])
    rows = []
    for p in path.rglob("*"):
        if p.is_file():
            rows.append({"extension": p.suffix.lower() or "[no extension]", "path": str(p)})
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=["extension", "count"])
    summary = df.groupby("extension").size().reset_index(name="count").sort_values("count", ascending=False)
    return summary

for name in ["thyus2path", "tn5000", "tn3k", "ddti", "tnui", "ddti_mirror"]:
    print(f"\n{name.upper()}")
    display(file_inventory(PATHS[name]))

## 2. Utility functions for image-mask audits

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".JPG", ".PNG"}


def image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in {e.lower() for e in IMAGE_EXTS}])


def stem_map(folder):
    return {p.stem: p for p in image_files(folder)}


def pair_images_masks(image_dir, mask_dir):
    images = stem_map(image_dir)
    masks = stem_map(mask_dir)
    common = sorted(set(images) & set(masks))
    missing_masks = sorted(set(images) - set(masks))
    missing_images = sorted(set(masks) - set(images))
    rows = [{"id": k, "image": str(images[k]), "mask": str(masks[k])} for k in common]
    return pd.DataFrame(rows), missing_masks, missing_images


def read_gray(path):
    return np.asarray(Image.open(path).convert("L"))


def mask_statistics(mask_path):
    arr = read_gray(mask_path)
    foreground = arr > 0
    return {
        "mask_height": arr.shape[0],
        "mask_width": arr.shape[1],
        "mask_foreground_pixels": int(foreground.sum()),
        "mask_area_fraction": float(foreground.mean()),
        "mask_unique_values": int(len(np.unique(arr))),
    }


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def hash_directory(folder, limit=None):
    rows = []
    files = image_files(folder)
    if limit is not None:
        files = files[:limit]
    for p in files:
        rows.append({"path": str(p), "basename": p.name, "stem": p.stem, "sha256": sha256_file(p)})
    return pd.DataFrame(rows)

## 3. TN3K audit

TN3K contains a train/validation image-mask directory and a test image-mask directory. We audit them separately and then check whether identical image hashes occur across the two splits.

In [ ]:
tn3k_pairs = []
for split, image_dir, mask_dir in [
    ("trainval", PATHS["tn3k"] / "trainval-image", PATHS["tn3k"] / "trainval-mask"),
    ("test", PATHS["tn3k"] / "test-image", PATHS["tn3k"] / "test-mask"),
]:
    pairs, missing_masks, missing_images = pair_images_masks(image_dir, mask_dir)
    pairs["dataset"] = "TN3K"
    pairs["split"] = split
    tn3k_pairs.append(pairs)
    print(f"{split}: paired={len(pairs)}, missing_masks={len(missing_masks)}, missing_images={len(missing_images)}")
    if missing_masks[:10]:
        print("  example missing masks:", missing_masks[:10])
    if missing_images[:10]:
        print("  example missing images:", missing_images[:10])

tn3k_manifest = pd.concat(tn3k_pairs, ignore_index=True) if tn3k_pairs else pd.DataFrame()
if not tn3k_manifest.empty:
    display(tn3k_manifest.groupby("split").size().rename("paired_count").to_frame())
    tn3k_manifest.to_csv(OUTPUT_ROOT / "tn3k_pairs.csv", index=False)

# Hash overlap check between TN3K trainval and test.
tn3k_hashes = []
for split in ["trainval", "test"]:
    sub = tn3k_manifest[tn3k_manifest["split"] == split] if not tn3k_manifest.empty else pd.DataFrame()
    for _, row in sub.iterrows():
        tn3k_hashes.append({"split": split, "id": row["id"], "sha256": sha256_file(row["image"])})
tn3k_hash_df = pd.DataFrame(tn3k_hashes)
if not tn3k_hash_df.empty:
    overlap_hashes = set(tn3k_hash_df.loc[tn3k_hash_df["split"] == "trainval", "sha256"]) & set(tn3k_hash_df.loc[tn3k_hash_df["split"] == "test", "sha256"])
    print("TN3K identical image hashes across trainval/test:", len(overlap_hashes))
    if overlap_hashes:
        display(tn3k_hash_df[tn3k_hash_df["sha256"].isin(overlap_hashes)].sort_values("sha256"))

## 4. DDTI audit

The mounted DDTI directory contains both original data and preprocessed `stage1`/`stage2` data. We will use the original image-mask directory for the primary external audit. The preprocessed stages must not be counted as independent samples because they may contain transformed versions of the same cases.

In [ ]:
ddti_original_images = PATHS["ddti"] / "1_or_data" / "image"
ddti_original_masks = PATHS["ddti"] / "1_or_data" / "mask"
ddti_pairs, ddti_missing_masks, ddti_missing_images = pair_images_masks(ddti_original_images, ddti_original_masks)

print("DDTI original paired:", len(ddti_pairs))
print("DDTI missing masks:", len(ddti_missing_masks))
print("DDTI missing images:", len(ddti_missing_images))
if ddti_missing_masks[:10]: print("Example missing masks:", ddti_missing_masks[:10])
if ddti_missing_images[:10]: print("Example missing images:", ddti_missing_images[:10])

ddti_pairs["dataset"] = "DDTI"
ddti_pairs["split"] = "original"
if not ddti_pairs.empty:
    display(ddti_pairs.head())
    ddti_pairs.to_csv(OUTPUT_ROOT / "ddti_pairs.csv", index=False)

for stage in ["stage1", "stage2"]:
    p, mm, mi = pair_images_masks(
        PATHS["ddti"] / "2_preprocessed_data" / stage / "p_image",
        PATHS["ddti"] / "2_preprocessed_data" / stage / "p_mask",
    )
    print(f"DDTI {stage}: paired={len(p)}, missing_masks={len(mm)}, missing_images={len(mi)}")

## 5. TNUI-2021 audit

In [ ]:
tnui_pairs = []
for split in ["train", "val", "test"]:
    image_dir = PATHS["tnui"] / split / "images"
    mask_dir = PATHS["tnui"] / split / "masks"
    pairs, missing_masks, missing_images = pair_images_masks(image_dir, mask_dir)
    pairs["dataset"] = "TNUI-2021"
    pairs["split"] = split
    tnui_pairs.append(pairs)
    print(f"{split}: paired={len(pairs)}, missing_masks={len(missing_masks)}, missing_images={len(missing_images)}")

tnui_manifest = pd.concat(tnui_pairs, ignore_index=True) if tnui_pairs else pd.DataFrame()
if not tnui_manifest.empty:
    display(tnui_manifest.groupby("split").size().rename("paired_count").to_frame())
    tnui_manifest.to_csv(OUTPUT_ROOT / "tnui_pairs.csv", index=False)

# Inspect the available classification label files.
for csv_name in ["train_img_labels.csv", "val_img_labels.csv", "test_img_labels.csv"]:
    csv_path = PATHS["tnui"] / csv_name
    print(f"\n{csv_name} exists={csv_path.exists()}")
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        print("shape:", df.shape)
        display(df.head())
        print("columns:", list(df.columns))

## 6. TN5000 XML and official split audit

TN5000 uses Pascal VOC-style XML annotations. This cell extracts:

- image ID;
- train/validation/test split;
- class name;
- bounding-box coordinates;
- image dimensions.

The parser is intentionally defensive because XML class names can differ slightly between dataset versions.

In [ ]:
def read_split_ids(split_file):
    if not split_file.exists():
        return []
    return [line.strip().split()[0] for line in split_file.read_text().splitlines() if line.strip()]


def parse_tn5000_xml(xml_path):
    out = {"xml": str(xml_path), "image_id": xml_path.stem, "class_name": None, "label": None,
           "xmin": None, "ymin": None, "xmax": None, "ymax": None,
           "xml_width": None, "xml_height": None}
    try:
        root = ET.parse(xml_path).getroot()
        size = root.find("size")
        if size is not None:
            out["xml_width"] = int(float(size.findtext("width", default="0")))
            out["xml_height"] = int(float(size.findtext("height", default="0")))
        obj = root.find("object")
        if obj is not None:
            raw_name = obj.findtext("name", default="")
            name = str(raw_name).strip()
            out["class_name"] = name
            name_lower = name.lower()
            # TN5000 XML versions may encode classes as 0/1 or as words.
            # The dataset convention is benign=0 and malignant=1.
            if name_lower in {"1", "malignant", "malign", "cancer", "positive"} or "malig" in name_lower:
                out["label"] = 1
            elif name_lower in {"0", "benign", "benign_nodule", "negative"} or "benign" in name_lower:
                out["label"] = 0
            box = obj.find("bndbox")
            if box is not None:
                for key in ["xmin", "ymin", "xmax", "ymax"]:
                    value = box.findtext(key)
                    out[key] = int(float(value)) if value is not None else None
    except Exception as e:
        out["parse_error"] = repr(e)
    return out


tn5000 = PATHS["tn5000"]
split_map = {}
for split in ["train", "val", "test", "trainval"]:
    ids = read_split_ids(tn5000 / "ImageSets" / "Main" / f"{split}.txt")
    for image_id in ids:
        split_map.setdefault(image_id, []).append(split)

xml_files = sorted((tn5000 / "Annotations").glob("*.xml")) if (tn5000 / "Annotations").exists() else []
tn5000_rows = []
for xml_path in xml_files:
    row = parse_tn5000_xml(xml_path)
    image_id = row["image_id"]
    image_path = None
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG"]:
        candidate = tn5000 / "JPEGImages" / f"{image_id}{ext}"
        if candidate.exists():
            image_path = candidate
            break
    row["image"] = str(image_path) if image_path else None
    split_list = split_map.get(image_id, [])
    row["split_list"] = ";".join(split_list)
    # Prefer the specific official split over the aggregate trainval file.
    # This produces the intended train/val/test counts.
    row["split"] = "test" if "test" in split_list else ("val" if "val" in split_list else ("train" if "train" in split_list else None))
    tn5000_rows.append(row)

tn5000_manifest = pd.DataFrame(tn5000_rows)
print("XML files:", len(xml_files))
print("Parsed manifest rows:", len(tn5000_manifest))
if not tn5000_manifest.empty:
    display(tn5000_manifest.head())
    print("Class names:")
    display(tn5000_manifest["class_name"].value_counts(dropna=False).to_frame("count"))
    print("Parsed labels:")
    display(tn5000_manifest["label"].value_counts(dropna=False).to_frame("count"))
    print("Splits:")
    display(tn5000_manifest["split"].value_counts(dropna=False).to_frame("count"))
    print("Missing image files:", int(tn5000_manifest["image"].isna().sum()))
    tn5000_manifest.to_csv(OUTPUT_ROOT / "tn5000_xml_manifest.csv", index=False)

# Check for IDs appearing in more than one official split.
multiple_split_ids = {k: v for k, v in split_map.items() if len(set(v)) > 1}
print("TN5000 IDs appearing in multiple split files:", len(multiple_split_ids))
if multiple_split_ids:
    print(dict(list(multiple_split_ids.items())[:20]))

## 7. ThyUS2Path CSV audit

ThyUS2Path contains multiple images per case. We inspect the metadata columns before deciding how to construct patient-level or case-level bags.

In [ ]:
def inspect_thyus2path_batch(batch_name):
    batch_root = PATHS["thyus2path"] / batch_name / batch_name
    print(f"\n{batch_name}: {batch_root} exists={batch_root.exists()}")
    if not batch_root.exists():
        return {}
    result = {}
    for csv_name in [f"{batch_name}.csv", f"{batch_name}_label.csv"]:
        csv_path = batch_root / csv_name
        print(f"  {csv_name}: exists={csv_path.exists()}")
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            result[csv_name] = df
            print("  shape:", df.shape)
            print("  columns:", list(df.columns))
            display(df.head())
    image_dir = batch_root / "dataset"
    print("  image count:", len(image_files(image_dir)))
    return result

thyus2path_tables = {}
thyus2path_tables["batch1_image"] = inspect_thyus2path_batch("batch1_image")
thyus2path_tables["batch2_image"] = inspect_thyus2path_batch("batch2_image")

## 8. Image and mask statistics

In [ ]:
def collect_pair_stats(manifest, max_images=None):
    if manifest is None or manifest.empty:
        return pd.DataFrame()
    rows = []
    subset = manifest.sample(min(max_images, len(manifest)), random_state=42) if max_images else manifest
    for _, row in subset.iterrows():
        try:
            image = Image.open(row["image"])
            image_arr = np.asarray(image.convert("L"))
            mask_stats = mask_statistics(row["mask"])
            rows.append({
                "dataset": row.get("dataset"),
                "split": row.get("split"),
                "id": row.get("id"),
                "image_height": image_arr.shape[0],
                "image_width": image_arr.shape[1],
                "image_channels": len(image.getbands()),
                **mask_stats,
            })
        except Exception as e:
            rows.append({"dataset": row.get("dataset"), "split": row.get("split"), "id": row.get("id"), "error": repr(e)})
    return pd.DataFrame(rows)

all_seg_manifests = []
if not tn3k_manifest.empty: all_seg_manifests.append(tn3k_manifest)
if not tnui_manifest.empty: all_seg_manifests.append(tnui_manifest)
if not ddti_pairs.empty: all_seg_manifests.append(ddti_pairs)

stats = []
for manifest in all_seg_manifests:
    stats.append(collect_pair_stats(manifest, max_images=500))
stats_df = pd.concat(stats, ignore_index=True) if stats else pd.DataFrame()
if not stats_df.empty:
    display(stats_df.groupby(["dataset", "split"])[["image_height", "image_width", "mask_area_fraction"]].describe())
    stats_df.to_csv(OUTPUT_ROOT / "segmentation_image_mask_statistics.csv", index=False)

## 9. Visual quality check

The following plots should be reviewed manually. We are looking for:

- mask/image misalignment;
- masks with zero foreground pixels;
- masks covering the wrong lesion;
- extreme borders or text overlays;
- incorrect class labels;
- large differences in image orientation or resolution.

In [ ]:
def show_pairs(manifest, n=6, title=""):
    if manifest is None or manifest.empty:
        print("No pairs available")
        return
    sample = manifest.sample(min(n, len(manifest)), random_state=42).reset_index(drop=True)
    fig, axes = plt.subplots(len(sample), 3, figsize=(12, 4 * len(sample)))
    if len(sample) == 1:
        axes = np.expand_dims(axes, axis=0)
    fig.suptitle(title, fontsize=16)
    for r, (_, row) in enumerate(sample.iterrows()):
        image = np.asarray(Image.open(row["image"]).convert("RGB"))
        mask = read_gray(row["mask"]) > 0
        axes[r, 0].imshow(image)
        axes[r, 0].set_title(f"Image: {row['id']}")
        axes[r, 1].imshow(mask, cmap="gray")
        axes[r, 1].set_title("Mask")
        axes[r, 2].imshow(image)
        axes[r, 2].imshow(mask, cmap="jet", alpha=0.35)
        axes[r, 2].set_title("Overlay")
        for c in range(3):
            axes[r, c].axis("off")
    plt.tight_layout()
    plt.show()

if not tn3k_manifest.empty:
    show_pairs(tn3k_manifest[tn3k_manifest["split"] == "trainval"], title="TN3K trainval samples")
if not tnui_manifest.empty:
    show_pairs(tnui_manifest[tnui_manifest["split"] == "train"], title="TNUI-2021 train samples")
if not ddti_pairs.empty:
    show_pairs(ddti_pairs, title="DDTI original samples")

## 10. Cross-dataset duplicate checks

Exact hash overlap can reveal leakage when a dataset mirror or transformed copy is present in multiple mounted folders. This is not a complete near-duplicate detector, but it is a necessary first check.

In [ ]:
def manifest_hashes(manifest, path_col="image", max_images=None):
    if manifest is None or manifest.empty:
        return pd.DataFrame(columns=["dataset", "split", "id", "path", "sha256"])
    subset = manifest.sample(min(max_images, len(manifest)), random_state=42) if max_images else manifest
    rows = []
    for _, row in subset.iterrows():
        path = row.get(path_col)
        if path and Path(path).exists():
            rows.append({
                "dataset": row.get("dataset"),
                "split": row.get("split"),
                "id": row.get("id"),
                "path": path,
                "sha256": sha256_file(path),
            })
    return pd.DataFrame(rows)

hash_tables = []
for manifest in all_seg_manifests:
    hash_tables.append(manifest_hashes(manifest, max_images=None))
hash_df = pd.concat(hash_tables, ignore_index=True) if hash_tables else pd.DataFrame()
if not hash_df.empty:
    dup_hashes = hash_df[hash_df.duplicated("sha256", keep=False)].sort_values("sha256")
    print("Exact duplicate image files across segmentation manifests:", len(dup_hashes))
    if not dup_hashes.empty:
        display(dup_hashes.head(50))
    hash_df.to_csv(OUTPUT_ROOT / "segmentation_image_hashes.csv", index=False)

## 12. Corrected TN5000 split and class audit

The first audit version parsed TN5000 XML class values as strings such as `0` and `1`. The corrected parser maps `0=benign` and `1=malignant` and uses the specific `train`, `val`, and `test` lists rather than counting the aggregate `trainval` list as a separate split.

In [ ]:
if not tn5000_manifest.empty:
    print("Corrected TN5000 class counts")
    display(tn5000_manifest["label"].value_counts(dropna=False).rename("count").to_frame())
    print("Corrected official split counts")
    display(tn5000_manifest["split"].value_counts(dropna=False).rename("count").to_frame())
    print("Unparsed labels:", int(tn5000_manifest["label"].isna().sum()))
    print("Image files missing:", int(tn5000_manifest["image"].isna().sum()))
    tn5000_manifest.to_csv(OUTPUT_ROOT / "tn5000_xml_manifest_corrected.csv", index=False)

## 13. Exact duplicate policy and clean manifests

Exact duplicate image hashes are removed from the evaluation manifests. The priority order is train, validation, test, then external/original. This prevents a duplicated validation or test image from appearing after the same image has already been used for model development.

The original manifests are retained. The cleaned manifests are used for training and evaluation only after the dropped rows have been documented.

In [ ]:
CLEAN_ROOT = OUTPUT_ROOT / "clean_manifests"
CLEAN_ROOT.mkdir(parents=True, exist_ok=True)

def add_hashes(manifest):
    if manifest is None or manifest.empty:
        return manifest.copy()
    out = manifest.copy()
    out["sha256"] = out["image"].apply(lambda p: sha256_file(p) if Path(p).exists() else None)
    return out

def clean_by_hash(manifest, priority):
    if manifest is None or manifest.empty:
        return manifest.copy(), manifest.copy()
    out = manifest.copy()
    out["_priority"] = out["split"].map(priority).fillna(99)
    out = out.sort_values(["_priority", "id"], kind="stable")
    out["duplicate_group"] = out["sha256"].map(out["sha256"].value_counts())
    keep = ~out.duplicated("sha256", keep="first")
    kept = out[keep].drop(columns=["_priority"])
    dropped = out[~keep].drop(columns=["_priority"])
    return kept, dropped

clean_records = []
dropped_records = []
for name, manifest in [("TN3K", tn3k_manifest), ("TNUI-2021", tnui_manifest), ("DDTI", ddti_pairs)]:
    if manifest is None or manifest.empty:
        continue
    m = add_hashes(manifest)
    if name == "TN3K":
        priority = {"trainval": 0, "test": 1}
    elif name == "TNUI-2021":
        priority = {"train": 0, "val": 1, "test": 2}
    else:
        priority = {"original": 0}
    kept, dropped = clean_by_hash(m, priority)
    kept.to_csv(CLEAN_ROOT / f"{name.lower().replace('-', '_')}_clean.csv", index=False)
    if not dropped.empty:
        dropped.to_csv(CLEAN_ROOT / f"{name.lower().replace('-', '_')}_dropped_duplicates.csv", index=False)
    clean_records.append(kept)
    dropped_records.append(dropped)
    print(f"{name}: original={len(m)}, kept={len(kept)}, dropped={len(dropped)}")
    if not dropped.empty:
        display(dropped[["dataset", "split", "id", "sha256"]])

all_dropped = pd.concat(dropped_records, ignore_index=True) if dropped_records else pd.DataFrame()
if not all_dropped.empty:
    all_dropped.to_csv(CLEAN_ROOT / "all_dropped_duplicates.csv", index=False)

## 14. Mask-value audit

TN3K masks are stored as JPEG images and may contain more than two grayscale values because of compression. Before training, compare thresholds such as `>0`, `>127`, and `>250`. The primary mask binarisation should be selected using visual overlays, not only the number of unique pixel values.

In [ ]:
def mask_value_report(manifest, n=20):
    if manifest is None or manifest.empty:
        return pd.DataFrame()
    sample = manifest.sample(min(n, len(manifest)), random_state=42)
    rows=[]
    for _, row in sample.iterrows():
        arr = read_gray(row["mask"])
        rows.append({
            "dataset": row["dataset"], "split": row["split"], "id": row["id"],
            "min": int(arr.min()), "max": int(arr.max()),
            "p01": float(np.percentile(arr, 1)), "p50": float(np.percentile(arr, 50)),
            "p99": float(np.percentile(arr, 99)),
            "unique_count": int(np.unique(arr).size),
            "positive_gt0": float((arr > 0).mean()),
            "positive_gt127": float((arr > 127).mean()),
            "positive_gt250": float((arr > 250).mean()),
        })
    return pd.DataFrame(rows)

for name, manifest in [("TN3K", tn3k_manifest), ("TNUI-2021", tnui_manifest), ("DDTI", ddti_pairs)]:
    print("\n", name)
    display(mask_value_report(manifest, n=20))

## 11. Preliminary decision log

After running the notebook, fill in the following decisions before training:

- [ ] Which TN3K split will be used for meta-training and internal testing?
- [ ] Will TNUI-2021 be an external segmentation test set or a source domain?
- [ ] Which DDTI version will be used, if any?
- [ ] Are there any exact duplicates across datasets or splits?
- [ ] Which patient/case identifiers are available?
- [ ] Are all mask/image pairs spatially aligned?
- [ ] Which classification labels are pathology-confirmed?
- [ ] Which dataset will remain untouched as the final external test set?
- [ ] Are the support/query episodes patient- or nodule-disjoint?

The next notebook should not be started until these decisions are recorded.